# 04 · Non-covalent contacts

rxembed can hold contacts between molecules while it embeds them. xyzgraph decides which atoms can make each kind
of contact, and `rx.contacts` groups compatible contacts into binding modes, each labelled by its contacts.
`contacts=` says which to hold:

| to hold | pass |
|---|---|
| every mode rxembed proposes | `contacts="auto"` |
| the modes of the kinds you choose | `contacts=rx.contacts(mol, kinds=("HB", "CHPI"))` |
| one mode | `contacts=modes[label]` |
| the contacts an input geometry makes | `contacts="input"` |
| a distance window of your own | `constrain={(i, j): (lo, hi)}` |

Each render draws the contacts xyzgraph detects in the result, dotted.

In [ ]:
import rxembed as rx
import xyzrender

## Every proposed mode

`contacts="auto"` embeds every binding mode and returns an `EnsembleSet`, one `Ensemble` per mode, named in
`.tag["nci"]`. Acetic acid and 2-aminopyridine have five; one is the ring of two H-bonds, O-H...N and N-H...O=C,
known from their co-crystals.

In [ ]:
PAIR = "CC(=O)O.Nc1ccccn1"  # acetic acid and 2-aminopyridine
modes = rx.embed(PAIR, contacts="auto", n=4, seed=1).minimize()
for mode in modes:
    print(mode.tag["nci"])
    display(xyzrender.render(xyzrender.load(mode.lowest(1).mol, nci_detect=True)))

## Choose the kinds

`rx.contacts` lists the modes before anything is embedded. With no `kinds` they are the ones `"auto"` embeds, built
from H-bonds (HB), halogen, chalcogen and pnictogen bonds (XB, ChB, PnB), ion pairs (IONIC) and cation lone pairs
(CATLP). `kinds=` builds them from the kinds named instead, and ring contacts (CHPI, HBPI, CATPI, ANPI, HALPI) join
only when named. Here CHPI adds the methyl over the ring to each H-bond mode.

In [ ]:
pair = rx.parse_smiles(PAIR)  # contacts name atoms of a Mol you hold
print(list(rx.contacts(pair)))  # what "auto" embeds
print(list(rx.contacts(pair, kinds=("HB", "CHPI"))))

Passing that result to `contacts=` embeds each of its modes. Each kind has its own distance window and, if
directional, an angle: a halogen bond (XB) along C-I...N, a chalcogen bond (ChB) along C-S...O, an ionic contact
(IONIC) and a cation over a ring face (CATPI), held to every ring atom. `binding_modes()` counts the kinds found
after relaxation.

In [ ]:
SYSTEMS = {
    "XB": "FC(F)(F)C(F)(F)I.c1ccncc1",  # pentafluoroiodoethane and pyridine
    "ChB": "c1nc2ccccc2s1.CC(C)=O",  # benzothiazole and acetone
    "IONIC": "C[N+](C)(C)C.[Cl-]",  # tetramethylammonium chloride
    "CATPI": "C[N+](C)(C)C.c1ccccc1",  # tetramethylammonium and benzene
}
for kind, smiles in SYSTEMS.items():
    mol = rx.parse_smiles(smiles)
    for bound in rx.embed(mol, contacts=rx.contacts(mol, kinds=(kind,)), n=4, seed=1).minimize():
        print(bound.tag["nci"], dict(bound.binding_modes()))
        display(xyzrender.render(xyzrender.load(bound.lowest(1).mol, nci_detect=True)))

## One mode

Pass one value of `rx.contacts`, picked by its label, to embed that mode alone as one `Ensemble`. Schreiner's
thiourea holds acetone's O with both N-H. `rx.contacts(mol, observed=True, conf_id=cid)` lists the contacts each
conformer makes without moving it.

In [ ]:
SCHREINER = "FC(F)(F)c1cc(cc(c1)C(F)(F)F)NC(=S)Nc1cc(cc(c1)C(F)(F)F)C(F)(F)F.CC(C)=O"
clamp = rx.parse_smiles(SCHREINER)
modes = rx.contacts(clamp)
print(list(modes))

bound = rx.embed(clamp, contacts=modes["HB:N17,N14->O35"], n=6, seed=1).minimize()
print(bound.tag["nci"])  # the label it was picked by
for cid in bound.ids:
    print(cid, list(rx.contacts(bound.mol, observed=True, conf_id=cid)))
xyzrender.render(xyzrender.load(bound.lowest(1).mol, nci_detect=True))

## Contacts from a geometry

`contacts="input"` holds the HB, XB, ChB, PnB, IONIC and CATLP contacts between fragments that the first conformer
makes, never pi contacts, and logs at INFO each pi contact it leaves. Here tetramethylammonium chloride sits over a
benzene ring, built from its IONIC and CATPI modes: the input holds the ion pair and leaves the cation-pi.

In [ ]:
SALT = "C[N+](C)(C)C.[Cl-].c1ccccc1"  # tetramethylammonium chloride and benzene
salt = rx.parse_smiles(SALT)
(ionic,) = rx.contacts(salt, kinds=("IONIC",)).values()
(cation_pi,) = rx.contacts(salt, kinds=("CATPI",)).values()
start = rx.embed(salt, contacts=(*ionic, *cation_pi), n=1, seed=1).minimize().mol

from_input = rx.embed(start, contacts="input", n=4, seed=1).minimize()
print(from_input.tag["nci"])

`rx.contacts(mol, observed=True)` lists every contact a geometry makes, pi contacts included, and holding a
chosen subset of them keeps those. Here it holds the cation-pi as well as the ion pair.

In [ ]:
found = rx.contacts(start, observed=True)
print(list(found))

both = rx.embed(start, contacts=(*found["IONIC:N1->Cl5"], *found["CATPI:N1->pi(6,7,8,9,10,11)"]), n=4, seed=1)
both.minimize()
print(dict(both.binding_modes()))
xyzrender.render(xyzrender.load(both.lowest(1).mol, nci_detect=True))

## A window of your own

`constrain={(i, j): (lo, hi)}` holds a distance you choose, pulled toward the middle of its window as a contact is.
A contact also places its H and angle: the H-bond modes above hold H...N at 1.6 to 2.2 angstrom with O-H...N within
40 degrees of linear, where this holds only the O...N distance. `rx.match` names the atoms.

In [ ]:
mol = rx.parse_smiles(PAIR)
acid_o, ring_n = rx.match(mol, "[OX2H].[nX2]")
window = rx.embed(mol, constrain={(acid_o, ring_n): (2.6, 2.9)}, n=4, seed=1)
print(f"O...N {window.measure((acid_o, ring_n))['mean']:.2f} A")
xyzrender.render(xyzrender.load(window.lowest(1).mol, nci_detect=True))